In [2]:
import os
import importlib
from openai import OpenAI
import scraper_hans

importlib.reload(scraper_hans)
from scraper_hans import fetch_website_contents, fetch_website_links_all

In [3]:
from static_data import *
Model=Llama_Model
Olama_Base_Url = Ollama_BASE_URL
API_Key=Api_key

In [4]:
openai = OpenAI(base_url=Olama_Base_Url, api_key=API_Key)

In [5]:
def website_content(url):
    contents = fetch_website_contents(url)
    print(contents)
    return contents

In [6]:
cnn_url="https://cnn.com"
content = website_content(cnn_url)


Breaking News, Latest News and Videos | CNN

CNN values your feedback
1. How relevant is this ad to you?
2. Did you encounter any technical issues?
Video player was slow to load content
Video content never loaded
Ad froze or did not finish loading
Video content did not start after ad
Audio on ad was too loud
Other issues
Ad never loaded
Ad prevented/slowed the page from loading
Content moved around while ad loaded
Ad was repetitive to ads I've seen previously
Other issues
Cancel
Submit
Thank You!
Your effort and contribution in providing this feedback is much
                                        appreciated.
Close
Ad Feedback
Close icon
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Ukraine-Russia War
Israel-Hamas War
Games
More
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Ukraine-Russia War
Israel-Hamas War
Games
Watch
Listen
Live TV
Subscribe
Sign in
My Account
Settings
Newsletters
Topics y

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example and it must have json with key 'type' amd 'url'. The json object must also have the key "links" and "links" key then have the list of dictionaries with  "type" ans "url" key:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """
    links=fetch_website_links_all(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [9]:
print(get_links_user_prompt(cnn_url))


    Here is the list of links on the website https://cnn.com -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    https://edition.cnn.com
https://edition.cnn.com/us
https://edition.cnn.com/world
https://edition.cnn.com/politics
https://edition.cnn.com/business
https://edition.cnn.com/health
https://edition.cnn.com/entertainment
https://edition.cnn.com/style
https://edition.cnn.com/travel
https://edition.cnn.com/sports
https://edition.cnn.com/science
https://edition.cnn.com/climate
https://edition.cnn.com/weather
https://edition.cnn.com/world/europe/ukraine
https://edition.cnn.com/world/middleeast/israel
https://edition.cnn.com/games
https://edition.cnn.com/us
https://edition.cnn.com/world
https://edition.cnn.com/politics
https://edition.cnn.com/business
https://edition.cnn.com/health
https

In [10]:
import json
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=Llama_Model,
        messages=[
            {
                "role":"system", "content": link_system_prompt
            },
            {
                "role":"user", "content": get_links_user_prompt(url)
            }
        ],
            response_format = {"type":"json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    print(result)
    return links


In [11]:
select_relevant_links(cnn_url)

{

 "url": "https://edition.cnn.com/"

}


{'url': 'https://edition.cnn.com/'}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {Llama_Model}")
    response = openai.chat.completions.create(
        model=Llama_Model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print("Links:", links)
    if 'links' in  links:
        print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links(cnn_url)

Selecting relevant links for https://cnn.com by calling llama3.2
Links: {}


{}

In [14]:
# select_relevant_links("https://huggingface.co")
select_relevant_links("https://nextjs.org/")

Selecting relevant links for https://nextjs.org/ by calling llama3.2
Links: {'links': [{'type': 'About page', 'url': 'https://vercel.com/about'}, {'type': 'Company page', 'url': 'https://vercel.com/about'}, {'type': 'Contact us page', 'url': 'https://vercel.com/contact/sales'}, {'type': 'Solutions page', 'url': 'https://vercel.com/solutions/nextjs'}, {'type': 'About our company', 'url': 'https://vercel.com/about'}, {'type': 'Team page', 'url': 'https://vercel.com/team'}, {'type': 'Blog', 'url': 'https://vercel.com/blog'}, {'type': 'GitHub repository', 'url': 'https://github.com/vercel/next.js'}, {'type': 'Blog', 'url': 'https://vercel.com/blog/next-16'}, {'type': 'Showcase', 'url': 'https://vercel.com/showcase'}, {'type': 'FAQ', 'url': 'https://vercel.com/support-policy'}, {'type': 'Governing documents', 'url': 'https://vercel.com/legal/privacy-policy'}, {'type': 'Customer success stories', 'url': 'https://vercel.com/customers/washington-post-next.js-vercel-engineering-at-the-speed-of-

{'links': [{'type': 'About page', 'url': 'https://vercel.com/about'},
  {'type': 'Company page', 'url': 'https://vercel.com/about'},
  {'type': 'Contact us page', 'url': 'https://vercel.com/contact/sales'},
  {'type': 'Solutions page', 'url': 'https://vercel.com/solutions/nextjs'},
  {'type': 'About our company', 'url': 'https://vercel.com/about'},
  {'type': 'Team page', 'url': 'https://vercel.com/team'},
  {'type': 'Blog', 'url': 'https://vercel.com/blog'},
  {'type': 'GitHub repository', 'url': 'https://github.com/vercel/next.js'},
  {'type': 'Blog', 'url': 'https://vercel.com/blog/next-16'},
  {'type': 'Showcase', 'url': 'https://vercel.com/showcase'},
  {'type': 'FAQ', 'url': 'https://vercel.com/support-policy'},
  {'type': 'Governing documents',
   'url': 'https://vercel.com/legal/privacy-policy'},
  {'type': 'Customer success stories',
   'url': 'https://vercel.com/customers/washington-post-next.js-vercel-engineering-at-the-speed-of-breaking-news'},
  {'type': 'Customer success 

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        print('Link found:',link)
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://nextjs.org"))
# print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://nextjs.org by calling llama3.2
Links: {'links': [{'type': 'Company Page', 'url': 'https://vercel.com'}, {'type': 'About Page', 'url': 'https://vercel.com/about'}, {'type': 'Contact/Sales Page', 'url': 'https://vercel.com/contact/sales'}, {'type': 'Documentation', 'url': 'https://vercel.com/docs'}, {'type': 'Blog', 'url': 'https://vercel.com/blog'}, {'type': 'Community Forum', 'url': 'https://community.vercel.com'}, {'type': 'GitHub Repository', 'url': 'https://github.com/vercel/next.js'}, {'type': 'GitHub Releases', 'url': 'https://github.com/vercel/next.js/releases'}, {'type': 'Showcase', 'url': 'https://vercel.com/showcase'}, {'type': 'Templates/Next.js', 'url': 'https://vercel.com/templates/next.js'}, {'type': 'New', 'url': 'https://vercel.com/new'}, {'type': 'Analytics', 'url': 'https://vercel.com/analytics'}, {'type': 'Previews', 'url': 'https://vercel.com/products/previews'}, {'type': 'Evaluations', 'url': 'https://vercel.com/evals'}, {'type':

In [17]:
brochure_system_prompt = """
You are an lazy lady assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("Nextjs", "https://nextjs.org")

Selecting relevant links for https://nextjs.org by calling llama3.2
Links: {'links': [{'type': 'about page', 'url': 'https://vercel.com/about'}, {'type': 'Company page', 'url': 'https://vercel.com/oss'}, {'type': 'Documentation/Getting Started', 'url': 'https://docs.vercel.com/'}, {'type': 'Documentation/Learn', 'url': 'https://docs.vercel.com/learn'}, {'type': 'Documentation/Showcase', 'url': 'https://docs.vercel.com/showcase'}, {'type': 'Blog', 'url': 'https://vercel.com/blog/'}, {'type': 'GitHub Repository', 'url': 'https://github.com/vercel/next.js'}, {'type': 'GitHub Releases', 'url': 'https://github.com/vercel/next.js/releases'}, {'type': 'Analytics', 'url': 'https://vercel.com/analytics'}, {'type': 'Contact/Sales', 'url': 'https://vercel.com/contact/sales'}, {'type': 'Community', 'url': 'https://community.vercel.com'}, {'type': 'FAQ/Governance', 'url': 'https://vercel.com/governance'}, {'type': 'FAQ/Ecosystem', 'url': 'https://vercel.com/ecosystem'}, {'type': 'FAQ/Working Group'

"\nYou are looking at a company called: Nextjs\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nNext.js by Vercel - The React Framework\n\nDocs\nLearn\nShowcase\nBlog\nTemplates\nDeploy\nSearch…\n⌘K\nAsk AI\nAsk AI\nSearch…\n⌘K\nDocs\nLearn\nShowcase\nBlog\nTemplates\nDeploy\nGitHub\nVercel OSS\nLatest\neve\nFrameworks\nNext.js\nSvelte\nNuxt\nNitro\nSDKs\nAI SDK\nChat SDK\nFlags SDK\nWorkflow SDK\nOther\nTurborepo\nshadcn/ui\nSWR\njust-bash\ndeepsec\nSelect a display theme:\nsystem\nlight\ndark\nThe React Framework for the Web\nUsed by the world's largest companies, Next.js enables you to create\nhigh-quality web applications\nwith the power of React components.\n$\nnpx create-next-app@latest\nnpx create-next-app@latest\nEverything you need to build great products on the web\nBuilt-in Optimizations\n→\nAutomatic Image, Font, and Script Optimizatio

In [20]:
from IPython.display import Markdown, display, update_display

def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=Llama_Model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("Nextjs", "https://nextjs.org")

Selecting relevant links for https://nextjs.org by calling llama3.2
Links: {'links': [{'type': 'about page', 'url': 'https://vercel.com/about'}, {'type': 'company page', 'url': 'https://vercel.com/solutions/nextjs'}, {'type': 'press/advertisement page', 'url': 'https://vercel.com/customers/washington-post-next.js-vercel-engineering-at-the-speed-of-breaking-news'}, {'type': 'press/advertisement page', 'url': 'https://vercel.com/customers/how-sonos-amplified-their-devex'}, {'type': 'careers page', 'url': 'https://github.com/vercel/next.js'}]}
Found 5 relevant links
Link found: {'type': 'about page', 'url': 'https://vercel.com/about'}
Link found: {'type': 'company page', 'url': 'https://vercel.com/solutions/nextjs'}
Link found: {'type': 'press/advertisement page', 'url': 'https://vercel.com/customers/washington-post-next.js-vercel-engineering-at-the-speed-of-breaking-news'}
Link found: {'type': 'press/advertisement page', 'url': 'https://vercel.com/customers/how-sonos-amplified-their-deve

**Welcome to Next.js**

Next.js is a modern React framework for building web applications. With a focus on ease of use, high-performance, and scalability, Next.js empowers developers to create high-quality web applications with the power of React components.

**At the Heart of Next.js**

Next.js is built on top of the React library, allowing developers to leverage the power of React components to build fast, scalable, and maintainable web applications. The framework includes a range of built-in optimizations, such as automatic image and font optimization, dynamic HTML streaming, and server-side rendering.

**Key Features**

* **Built-in Optimizations**: Next.js includes a range of built-in optimizations designed to improve the performance and user experience of web applications.
* **Server-Side Rendering**: Next.js supports server-side rendering, allowing developers to pre-render pages and improve SEO.
* **Static Site Generation**: Next.js also supports static site generation, enabling developers to pre-build and deploy static sites.
* **AI and Machine Learning**: Next.js includes a range of AI and machine learning-powered features, including an AI SDK and chat SDK.
* **Security**: Next.js includes a range of security features, including automatic security updates and bug fixes.

**Industry Leaders**

Next.js is used by some of the world's largest companies, including eVerge, Smyte, and Facebook. Backed by a range of individual investors, including Nat Friedman, Jordan Walke, and Naval Ravikant, Next.js is a platform that's shaping the future of web development.

**Join the Community**

Next.js has a thriving community of developers, with a range of resources available, including documentation, tutorials, and forums. Join the community today and start building with Next.js.

**Careers**

At Next.js, we're committed to creating a workplace that's supportive, inclusive, and empowering. We're currently hiring for a range of roles, including developers, engineers, and product managers. Join our team today and be a part of shaping the future of web development.

In [22]:
brochure_system_prompt = """
You are an ver smart and intelligent assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits, new features and clients.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [23]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [25]:
from IPython.display import Markdown, display, update_display

def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=Llama_Model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("Nextjs", "https://nextjs.org")

Selecting relevant links for https://nextjs.org by calling llama3.2
Links: {'links': [{'type': 'About page', 'url': 'https://vercel.com/about'}, {'type': 'Company page', 'url': 'https://vercel.com/about'}, {'type': 'Showcase page', 'url': 'https://vercel.com'}, {'type': 'Blog', 'url': 'https://vercel.com/blog'}, {'type': 'Jobs/Careers', 'url': 'https://github.com/vercel/next.js'}, {'type': 'Templates/Showcase', 'url': 'https://vercel.com/templates/next.js'}]}
Found 6 relevant links
Link found: {'type': 'About page', 'url': 'https://vercel.com/about'}
Link found: {'type': 'Company page', 'url': 'https://vercel.com/about'}
Link found: {'type': 'Showcase page', 'url': 'https://vercel.com'}
Link found: {'type': 'Blog', 'url': 'https://vercel.com/blog'}
Link found: {'type': 'Jobs/Careers', 'url': 'https://github.com/vercel/next.js'}
Link found: {'type': 'Templates/Showcase', 'url': 'https://vercel.com/templates/next.js'}


# Next.js: Building the Future of Web Applications

## About Us

Next.js is a popular React framework for building high-quality web applications. Founded by Vercel, Next.js is used by the world's largest companies to create stunning and performant web experiences. Our mission is to empower developers to build amazing apps with ease, using the latest React features and built-in optimizations.

## What We Do

Next.js offers a range of features that make it the ideal choice for building web applications. Some of our key features include:

* Automatic Image, Font, and Script Optimizations for improved UX and Core Web Vitals
* Dynamic HTML Streaming for instant UI rendering
* React Server Components for seamless integration with server-side rendering
* Data Fetching with both server and client-side options
* CSS Support for popular tools and libraries
* Client and Server Rendering with flexible rendering and caching options

## Our Technology

Next.js is built on top of React and is designed to take full advantage of the latest React features. We also offer a range of tools and APIs to help developers build and deploy their apps quickly and efficiently.

## Our Culture

At Next.js, we're passionate about empowering developers to build amazing apps. Our team is dedicated to helping developers succeed, with a focus on community, innovation, and collaboration.

## Customers

Next.js is used by some of the world's largest and most innovative companies, including:

* Airbnb
* Dropbox
* Instagram
* Shopify

## Join Our Mission

If you're looking for a new challenge, want to join a talented team of developers, or just want to be a part of something amazing, we want to hear from you! Check out our open positions and join our team!

## Get Started

Ready to build your next web application? Start with our easy-to-use tool, `npx create-next-app@latest`. Get started today and see what's possible with Next.js!